# AI Engineering Interview Prep
## Section: FastAPI

**Mangesh Jha | Senior Software Developer → AI Engineer**

---
> 💡 **How to use:** Use `Ctrl+F` to search any question. 🏷️ markers link to Mangesh's real project experience.


---
## ⚡ Section 18 — FastAPI

### Basics

### Q1. What is FastAPI?
**A:** A modern, high-performance web framework for building APIs with Python 3.8+ based on standard Python type hints. Built on top of Starlette (for web parts) and Pydantic (for data validation).

### Q2. Why FastAPI over Flask?
**A:** FastAPI provides native asynchronous (`async`/`await`) support, automatic data validation/serialization using Pydantic, and automatic interactive Swagger/ReDoc documentation generation, whereas Flask is synchronous by default and requires separate extensions for these features.

### Q3. Why FastAPI over Django?
**A:** FastAPI is lightweight, async-first, and highly optimized for building microservices and REST/GraphQL APIs. Django is a heavyweight, synchronous full-stack framework with built-in admin panels and ORM, which has more overhead.

### Q4. FastAPI Architecture?
**A:** FastAPI is an ASGI-compatible application layer built on **Starlette** (ASGI toolkit handling routing, sessions, WebSockets) and **Pydantic** (data validation/parsing layer). It uses **Uvicorn** as the underlying ASGI web server.

### Q5. ASGI vs WSGI
**A:** 
* **WSGI** (Web Server Gateway Interface): Synchronous protocol. It handles one request per thread/process at a time, making it bad for long-lived connections (WebSockets, streaming).
* **ASGI** (Asynchronous Server Gateway Interface): Successor to WSGI. It supports asynchronous execution, enabling a single process to handle WebSockets, SSE, HTTP streaming, and concurrent long-polling requests.

### API Development

### Q6. How do you create an endpoint?
**A:** 
```python
from fastapi import FastAPI
app = FastAPI()

@app.get("/")
def read_root():
    return {"hello": "world"}
```

### Q7. GET vs POST vs PUT vs PATCH vs DELETE
**A:** 
* `GET`: Retrieves data.
* `POST`: Submits data to create a new resource.
* `PUT`: Replaces an entire existing resource.
* `PATCH`: Partially updates an existing resource.
* `DELETE`: Deletes a resource.

### Q8. Path Parameters
**A:** Variable parts of the URL path used to identify a resource.
```python
@app.get("/users/{user_id}")
def get_user(user_id: int):
    return {"user_id": user_id}
```

### Q9. Query Parameters
**A:** Key-value pairs added after the `?` in a URL, usually optional and used for filtering/paging.
```python
@app.get("/items/")
def read_items(skip: int = 0, limit: int = 10):
    return {"skip": skip, "limit": limit}
```

### Q10. Request Body
**A:** The payload sent in the HTTP request (typically JSON), defined using a Pydantic model.
```python
from pydantic import BaseModel
class Item(BaseModel):
    name: str
    price: float
@app.post("/items/")
def create_item(item: Item):
    return item
```

### Pydantic

### Q11. What is Pydantic?
**A:** A data validation and settings management library using Python type annotations. It enforces type hints at runtime and provides user-friendly errors when data is invalid.

### Q12. Why does FastAPI use Pydantic?
**A:** To perform automatic request body parsing, runtime data validation, data sanitization/coercion, and response serialization.

### Q13. Data Validation Example
**A:** 
```python
from pydantic import BaseModel, EmailStr
class User(BaseModel):
    username: str
    email: EmailStr
    age: int
```

### Q14. Optional Fields
**A:** Defined using `Optional` from the `typing` module or `None` default values (or `str | None` in Python 3.10+).
```python
from typing import Optional
class User(BaseModel):
    bio: Optional[str] = None
```

### Q15. Nested Models
**A:** Pydantic models can contain lists or dicts of other Pydantic models.
```python
class Image(BaseModel):
    url: str
class Item(BaseModel):
    name: str
    images: list[Image]
```

### Q16. Field Validators
**A:** Custom validation logic applied to specific fields using the `@validator` decorator (or `@field_validator` in Pydantic v2).
```python
from pydantic import BaseModel, validator
class User(BaseModel):
    age: int
    @validator("age")
    def must_be_adult(cls, v):
        if v < 18: raise ValueError("Must be 18+")
        return v
```

### Q17. Custom Validation
**A:** Implemented using `@validator` (specific fields) or `@root_validator` (model-wide fields check, like password confirmation matching).

### Q18. Serialization
**A:** The process of converting Pydantic objects or database models back into standard JSON formats. In Pydantic, this is done via `model.dict()` (v1) or `model.model_dump()` (v2).

### Q19. Response Models
**A:** Defining the exact Pydantic model format that the endpoint is allowed to return.
```python
@app.get("/users/{id}", response_model=UserOut)
def read_user(id: int):
    return db_user
```

### Q20. Why Response Models?
**A:** It filters out private/unwanted fields (like hashed passwords), validates outgoing data, format-coerces fields, and automatically documents the response in Swagger.

### Dependency Injection

### Q21. What is Depends()?
**A:** A utility function in FastAPI used to declare a dependency. FastAPI will automatically evaluate the dependency function and inject its return value into the route handler.

### Q22. Why Dependency Injection?
**A:** It maximizes code reuse (e.g. sharing database session creation, JWT verification, query parameters parsing) and makes unit testing easier by allowing dependencies to be easily mocked.

### Q23. Database Dependency Example
**A:** 
```python
def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()
@app.get("/items/")
def read_items(db: Session = Depends(get_db)):
    return db.query(Item).all()
```

### Q24. Shared Dependencies
**A:** Dependencies that are declared globally or at the router level so that they apply to all endpoints (e.g. enforcing authentication on an entire set of routes).

### Q25. Dependency Lifecycle
**A:** A dependency runs when a request hits the endpoint. If it uses `yield` (like the DB session), it executes the setup code, yields the value to the endpoint, and executes the cleanup code (after `yield`) after the response is sent.

### Async Programming

### Q26. Difference between async def and def
**A:** 
* `async def`: Defines a coroutine that runs on the main event loop. It must yield execution via `await` when performing I/O.
* `def`: Defines a synchronous blocking function. FastAPI executes `def` functions on a background thread pool to prevent them from blocking the main event loop.

### Q27. When should async be used?
**A:** For I/O-bound tasks (network requests, calling database queries, sending messages to queues, reading/writing files) where the system spends time waiting.

### Q28. What happens internally in async endpoint?
**A:** When the async endpoint hits an `await` statement, it pauses execution and yields control back to the event loop. The event loop then runs other tasks until the awaited operation signals that it is complete.

### Q29. What is await?
**A:** A keyword that pauses execution of the current coroutine, returning control to the event loop until the awaited task completes and returns its result.

### Q30. Event Loop Explain
**A:** The core engine of async Python. It runs in a loop, scheduling tasks, checking if registered I/O events (like network packets arriving) are ready, and executing active coroutines.

### Q31. Async Database Calls
**A:** Using database drivers that support async (like `asyncpg` or `aiomysql` with `SQLAlchemy`'s `ext.asyncio` extensions) to execute database queries without blocking the event loop.

### Q32. Async HTTP Calls
**A:** Using non-blocking HTTP clients like `httpx` or `aiohttp` to call third-party APIs (like LLM providers) without stopping other requests.
```python
import httpx
async with httpx.AsyncClient() as client:
    response = await client.get("https://api.openai.com/...")
```

### Q33. Blocking vs Non-Blocking
**A:** 
* **Blocking**: A synchronous execution model where the thread is held and cannot perform any other work while waiting for an operation to finish.
* **Non-Blocking**: An asynchronous execution model where the thread is released to perform other tasks while waiting for the operation to complete.

### Q34. Why async matters in GenAI APIs?
**A:** AI APIs are highly I/O-bound: calls to LLM providers (which can take seconds to generate text) or document retrieval from vector databases can block the system. Async allows the server to handle thousands of simultaneous client connections and stream tokens concurrently.

### Q35. Async vs Threading
**A:** 
* **Async**: Single-threaded, cooperative multitasking. The developer decides when to switch contexts (via `await`). Low overhead, supports high concurrency.
* **Threading**: Multi-threaded, preemptive multitasking managed by the OS. Context switches are arbitrary, meaning there is thread overhead and potential for race conditions.

### Authentication

### Q36. JWT Authentication
**A:** JSON Web Token authentication. Users log in with credentials, the server generates a signed cryptographical token (JWT), and the client sends this token in the header (`Authorization: Bearer <token>`) for subsequent requests.

### Q37. OAuth2 Flow
**A:** The industry standard protocol for authorization. FastAPI integrates with the **Authorization Code Flow with PKCE** or **Resource Owner Password Credentials Flow** to safely authenticate clients and issue access tokens.

### Q38. Refresh Tokens
**A:** Long-lived tokens stored securely on the client side, used to request new short-lived access tokens from the auth server once they expire without prompting user re-login.

### Q39. Access Tokens
**A:** Short-lived cryptographic tokens (usually JWTs, valid for 15-30 minutes) used by the client to authorize API requests.

### Q40. How FastAPI handles security?
**A:** FastAPI provides the `fastapi.security` module which offers built-in classes (`OAuth2PasswordBearer`, `APIKeyHeader`, `HTTPBasic`) that extract credentials from headers/cookies and integrate directly into the dependency injection engine.

### Database

### Q41. FastAPI with SQLAlchemy
**A:** Implemented by declaring a database engine, creating a `sessionmaker`, and using a dependency function (`get_db`) to yield session instances into endpoints.

### Q42. ORM vs Raw SQL
**A:** 
* **ORM** (Object-Relational Mapper): Maps database tables to Python classes. Safer (prevents SQL Injection), faster to write, but can add query overhead.
* **Raw SQL**: Hand-written queries. Highly optimized, but tedious to write and maintain, and requires careful parameters parsing to prevent security risks.

### Q43. Session Management
**A:** Creating a database session per request and ensuring it is closed (`db.close()`) once the request completes, which is managed cleanly using a yielding FastAPI dependency.

### Q44. Connection Pooling
**A:** A cache of active database connections maintained by the server. Instead of creating and destroying connections for every request (which is slow), FastAPI/SQLAlchemy reuse connections from the pool.

### Q45. Async SQLAlchemy
**A:** Using `create_async_engine` and `AsyncSession` alongside an async driver (like `asyncpg`) to perform database queries in a non-blocking manner.

### Production Deployment

### Q46. How do you deploy FastAPI?
**A:** FastAPI applications are deployed inside Docker containers using an ASGI stack.
```
Client
  ↓
Nginx
  ↓
Gunicorn
  ↓
Uvicorn Workers
  ↓
FastAPI
```
Explanation: Nginx acts as the reverse proxy (handling SSL/static content), Gunicorn acts as the process manager, which spawns and runs Uvicorn workers (`UvicornWorker`) to execute the FastAPI application.

### Q47. What is Uvicorn?
**A:** A lightning-fast ASGI web server implementation for Python, built on `uvloop` (a fast C implementation of the event loop) and `httptools`.

### Q48. What is Gunicorn?
**A:** A WSGI HTTP server commonly used as a process manager in production to monitor, spawn, restart, and scale worker processes (like Uvicorn workers) to ensure high availability.

### Q49. How to Dockerize FastAPI?
**A:** 
```dockerfile
FROM python:3.10
WORKDIR /code
COPY ./requirements.txt /code/requirements.txt
RUN pip install --no-cache-dir --upgrade -r /code/requirements.txt
COPY ./app /code/app
CMD ["gunicorn", "app.main:app", "-w", "4", "-k", "uvicorn.workers.UvicornWorker", "-b", "0.0.0.0:80"]
```

### Q50. How to scale FastAPI for 10,000 concurrent users?
**A:** 
1. Set up horizontal scaling behind a Load Balancer (e.g. AWS ALB or Kubernetes Ingress).
2. Run FastAPI asynchronously using Uvicorn workers.
3. Implement Connection Pooling and Async database drivers to prevent database bottlenecks.
4. Use **Redis** for session management, semantic caching, and rate limiting.
5. Offload heavy computation (like document processing or LLM inference) to Celery task queues.

### FastAPI + GenAI Questions

### Q51. How would you expose an LLM through FastAPI?
**A:** Expose a POST endpoint that accepts parameters (prompt, temperature, etc.) via a Pydantic model, invokes the LLM client (e.g. OpenAI or Anthropic SDKs), and returns the generated text.
```python
@app.post("/generate")
async def generate_text(payload: PromptPayload):
    response = await openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": payload.prompt}]
    )
    return {"response": response.choices[0].message.content}
```

### Q52. How would you stream LLM responses?
**A:** By using `StreamingResponse` from `fastapi.responses` combined with an asynchronous generator function that streams chunks from the LLM provider.
```python
from fastapi.responses import StreamingResponse

async def token_generator(prompt):
    stream = await openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        stream=True
    )
    async for chunk in stream:
        content = chunk.choices[0].delta.content
        if content:
            yield content

@app.get("/stream")
async def stream_endpoint(prompt: str):
    return StreamingResponse(token_generator(prompt), media_type="text/event-stream")
```

### Q53. How would you upload PDFs for RAG?
**A:** Define a POST endpoint that accepts files via `UploadFile`, saves/reads the file stream, parses it using a PDF extractor, splits it into chunks, generates embeddings, and saves them to the vector store.
```python
from fastapi import UploadFile, File
@app.post("/upload-pdf")
async def upload_pdf(file: UploadFile = File(...)):
    contents = await file.read()
    # Parse PDF contents, chunk, embed, and store in FAISS/Pinecone
    return {"filename": file.filename, "status": "indexed"}
```

### Q54. How would you process large documents asynchronously?
**A:** By offloading the document processing to a background task using FastAPI's `BackgroundTasks` or a dedicated distributed task queue like **Celery** with Redis, preventing the HTTP request from timing out.
```python
from fastapi import BackgroundTasks

def process_pdf_task(file_path):
    # Heavy parsing, embedding, and indexing work
    pass

@app.post("/process-large-doc")
async def process_doc(background_tasks: BackgroundTasks):
    background_tasks.add_task(process_pdf_task, "path/to/doc.pdf")
    return {"message": "Document processing started in background"}
```

### Q55. How would you integrate LangChain with FastAPI?
**A:** By instantiating LangChain chains, runnables, or agents globally or inside dependency injection, and invoking them inside endpoints. For streaming, you can use LangChain's async streaming API (`astream` or `astream_events`) wrapped in a FastAPI `StreamingResponse`.

### Q56. How would you handle 1000 simultaneous AI requests?
**A:** 
1. Use fully asynchronous routing to avoid blocking thread pools.
2. Implement **Prompt Caching** (supported by Anthropic/OpenAI) to reduce latency and processing overhead.
3. Deploy FastAPI behind a load balancer on multiple containers.
4. Use a Redis-based rate limiter to protect endpoints from overload.
5. Implement request queuing/buffering.

### Q57. How would you cache LLM responses?
**A:** By storing prompt/response pairs in a caching layer. For exact match lookup, use standard key-value caching. For semantic similarity match lookup (handling slightly rephrased queries), implement **Semantic Caching** using vector similarity search.

### Q58. Redis vs Memory Cache?
**A:** 
* **Memory Cache**: Extremely fast, but local to a single container/process. It gets cleared on container restart and cannot be shared across multiple scaled instances.
* **Redis**: Distributed and persistent. It can be shared across all scaled instances of FastAPI, makes capacity planning easier, and supports semantic vector indices.

### Q59. How would you implement conversation history?
**A:** By storing conversation history (list of user/assistant messages) in **Redis** indexed by a unique `session_id`. When a request comes in, retrieve the history, append the new message, send it to the LLM, and update the store.
```python
@app.post("/chat")
async def chat(session_id: str, message: str):
    history = await redis.get(session_id) or []
    history.append({"role": "user", "content": message})
    response = await call_llm(history)
    history.append({"role": "assistant", "content": response})
    await redis.set(session_id, history)
    return {"response": response}
```

### Q60. How would you secure an AI API from prompt injection?
**A:** 
1. Use structured inputs (JSON schemas) instead of passing raw, unvalidated text blobs.
2. Enforce strict **system prompts** that instruct the model to ignore user attempts to override instructions.
3. Run an validation layer (like **Llama Guard** or **NeMo Guardrails**) to check inputs before sending them to the core LLM.
4. Implement input sanitization and length limits on user prompts.